In [1]:
import pandas as pd
import torch
import anndata as ad
import scanpy as sc
import os

In [2]:
file_path='/fs/ess/PAS1475/Xiaojie/graph_foundation_model'
save_path='/fs/ess/PAS1475/yzhong/sf_project/datasets/segmentation_noise'
os.makedirs(save_path, exist_ok=True)

In [3]:
adata_path = "/fs/ess/PAS1475/yzhong/sf_project/backup/ccv/datasets/CosMx"
adata_dict = torch.load(adata_path+"/CosMx_Human_Lung_processed.pt")
adata_dict

/tmp/ipykernel_671247/1444727105.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  adata_dict = torch.load(adata_path+"/CosMx_Human_Lung_processed.pt")


{'Lung5_Rep1': AnnData object with n_obs × n_vars = 98002 × 960
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden'
     var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
     uns: 'log1p', 'pca', 'neighbors', 'umap', 'leiden'
     obsm: 'X_pca', 'X_umap'
     varm: 'PCs'
     layers: 'counts'
     obsp: 'distances', 'connectivities',
 'Lung5_Rep2': AnnData object with n_obs × n_vars = 105800 × 960
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_

In [4]:
sample_name_l=["Lung5_Rep1","Lung5_Rep2","Lung5_Rep3","Lung6",
             "Lung9_Rep1","Lung9_Rep2","Lung12","Lung13"]
noise_level_l1=["0.5","1.0","2.0","5.0"]
noise_level_l2=["0.5","1","2","5"]

for sample_name in sample_name_l:
    new_adata_dict={}

    for noise_level1, noise_level2 in zip(noise_level_l1, noise_level_l2):

        adata=adata_dict[sample_name]
        sample_index=pd.unique(adata.obs['slide_ID_numeric'].values)[0].astype("str")

        sample_count=pd.read_csv(file_path+f"/{sample_name}/noisy_boundaries/noisy_expr_{noise_level1}.csv")
        sample_count.fillna(0, inplace=True)
        sample_count.drop_duplicates(subset=['fov', 'cell_ID'], keep='first', inplace=True)

        sample_coord=pd.read_csv(file_path+f"/{sample_name}/noisy_boundaries/centroid_{noise_level2}_new_global.csv")
        sample_count=pd.merge(sample_count, sample_coord, on=['fov', 'cell_ID'])
        sample_count.set_index('c_'+sample_index+'_'+sample_count['fov'].astype(str) + '_' + sample_count['cell_ID'].astype(str), inplace=True)

        adata=adata[adata.obs.index.isin(sample_count.index)]
        adata_new=ad.AnnData(X=sample_count.loc[adata.obs.index, adata.var.index],
                            obs=adata.obs,
                            var=adata.var)
        adata_new.obs[['new_x',"new_y"]]=sample_count.loc[adata.obs.index, ['x_global','y_global']]

        sc.pp.filter_cells(adata_new, min_counts=10)
        sc.pp.filter_genes(adata_new, min_cells=5)
        adata_new.layers["counts"] = adata_new.X.copy()
        sc.pp.normalize_total(adata_new, inplace=True)
        sc.pp.log1p(adata_new)

        new_adata_dict[sample_name+"_"+noise_level1]=adata_new

    common_cells = ad.concat(new_adata_dict, axis=1, join='inner').obs.index
    for adata_name, adata in new_adata_dict.items():
        new_adata_dict[adata_name]=adata[adata.obs.index.isin(common_cells)].copy()

    print(new_adata_dict)
    torch.save(new_adata_dict, save_path+f'/CosMx_Human_{sample_name}_noise_processed.pt')

/fs/ess/PAS1475/yzhong/sf_project/conda_env/G2PM/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


{'Lung5_Rep1_0.5': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung5_Rep1_1.0': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 

/fs/ess/PAS1475/yzhong/sf_project/conda_env/G2PM/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


{'Lung5_Rep2_0.5': AnnData object with n_obs × n_vars = 94131 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung5_Rep2_1.0': AnnData object with n_obs × n_vars = 94131 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 

/fs/ess/PAS1475/yzhong/sf_project/conda_env/G2PM/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

{'Lung9_Rep2_0.5': AnnData object with n_obs × n_vars = 127340 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung9_Rep2_1.0': AnnData object with n_obs × n_vars = 127340 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov'

/fs/ess/PAS1475/yzhong/sf_project/conda_env/G2PM/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


{'Lung12_0.5': AnnData object with n_obs × n_vars = 67085 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung12_1.0': AnnData object with n_obs × n_vars = 67085 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 

/fs/ess/PAS1475/yzhong/sf_project/conda_env/G2PM/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


{'Lung13_0.5': AnnData object with n_obs × n_vars = 76421 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung13_1.0': AnnData object with n_obs × n_vars = 76421 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 